# Denman Glacier — Grounding-line Linestring × MEaSUREs Velocity, indexed by Morton

This notebook shows a practical use of `mortie.linestring_coverage` and `mortie.morton_buffer`: using morton indices as the common key between two datasets on **completely different grids**, to pull out just the velocity pixels that lie within a small buffer around the grounding line — **without** any bbox-based spatial joins. Morton indices *are* the spatial index; the glacier name is the AOI selector.

**Datasets**

| # | Product | Grid | Temporal |
|---|---|---|---|
| 1 | [NSIDC-0498 v2](https://nsidc.org/data/nsidc-0498/versions/2) — MEaSUREs Antarctic Grounding Line (InSAR) | vector polylines | annual, 1992–2025 |
| 2 | [NSIDC-0761 v1](https://nsidc.org/data/nsidc-0761/versions/1) — MEaSUREs Multi-year Reference Velocity | raster, 450 m, EPSG:3031 | four multi-year epochs: **1995–2001, 2007–2009, 2014–2017, 2020–2022** |

**Pipeline**

1. AOI is selected by **name**, not bbox: grounding-line features are filtered by `Glac_Name == 'Denman'`. The continent-wide velocity rasters are downloaded and processed in full.
2. Grounding-line coverage at **order 17 (~50 m cells)** with a **k=1 buffer** (one-cell ring around the line).
3. Velocity pixels → morton indices at the coarser **match order** (14 ≈ 400 m, 15 ≈ 200 m) — computed across the full continent, in row-chunks so we never hold a full 100M-pixel morton grid at once.
4. Grounding-line order-17 morton indices are **clipped** to the match order by stripping trailing morton digits — digit truncation is equivalent to an ancestor lookup in the HEALPix nested hierarchy.
5. The **intersection** of the two sets at the match order is the set of velocity pixels within the buffered grounding line. This is the actual payload we display.
6. A time slider picks a grounding-line **year**. The velocity epoch is the one containing that year; if the GL year falls outside every epoch, we fall back to the **nearest** epoch. The plot title always shows both.

A `DENMAN_*` bbox is defined later — but it's used **only** as the cartopy plot extent, never as a filter on the data or the CMR queries. Morton-index matching is what localises the pixels.

In [ ]:
from pathlib import Path
import numpy as np
import xarray as xr
import geopandas as gpd
import pandas as pd
from pyproj import Transformer
from shapely.geometry import LineString, MultiLineString
from shapely.geometry import box as shapely_box
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import healpy as hp
import ipywidgets as widgets
from IPython.display import display

import earthaccess
import mortie
print('mortie', mortie.__version__)

## Helpers and constants

The Denman bbox below is used **only** to set the cartopy plot extent. Nothing else references it.

In [ ]:
# --- Plot extent (not a data filter) ------------------------------------
# Denman Glacier sits at ~100°E, ~66.5°S. This bbox is passed ONLY to
# cartopy's ax.set_extent(...). The CMR queries, the grounding-line
# feature filter, and the velocity morton indexing all run without it.
DENMAN_LONMIN, DENMAN_LONMAX = 98.0, 104.5
DENMAN_LATMIN, DENMAN_LATMAX =-67.5, -65.5

# Morton match orders — compared side-by-side.
MATCH_ORDERS = (14, 15)
# Grounding-line coverage order (fine).
GL_ORDER = 17
# Buffer cells around the line, in cells at GL_ORDER.
GL_BUFFER_K = 1

# Cell side lengths in meters (sqrt(pi/3) / nside * R_earth).
R_EARTH_M = 6_371_008.7714
def cell_width_m(order):
    return R_EARTH_M * np.sqrt(np.pi / 3.0) / (1 << order)

for o in (GL_ORDER, *MATCH_ORDERS):
    print(f'  order {o}: ~{cell_width_m(o):7.1f} m per cell side')

In [ ]:
def morton_clip_order(morton_indices, src_order, tgt_order):
    """Clip morton indices from src_order -> tgt_order by digit truncation.

    In mortie's encoding, each level of HEALPix nesting contributes one
    decimal digit to the morton index. Dropping the last (src-tgt) digits
    is equivalent to walking up the nested tree by that many levels.
    """
    if tgt_order == src_order:
        return np.asarray(morton_indices, dtype=np.int64)
    if tgt_order > src_order:
        raise ValueError('tgt_order must be <= src_order')
    m = np.asarray(morton_indices, dtype=np.int64)
    factor = 10 ** (src_order - tgt_order)
    out = np.abs(m) // factor
    out = np.where(m < 0, -out, out).astype(np.int64)
    return out


def shapely_to_latlon(geom):
    """Convert LineString / MultiLineString -> (lats, lons) per-line arrays.

    Always returns (list_of_lats, list_of_lons) so the caller can extend
    without branching.
    """
    if isinstance(geom, LineString):
        xs, ys = np.asarray(geom.xy[0]), np.asarray(geom.xy[1])
        return [ys], [xs]
    if isinstance(geom, MultiLineString):
        las, los = [], []
        for ls in geom.geoms:
            xs, ys = np.asarray(ls.xy[0]), np.asarray(ls.xy[1])
            las.append(ys); los.append(xs)
        return las, los
    raise TypeError(type(geom).__name__)

## 1. Grounding lines — NSIDC-0498, filtered to `Glac_Name == 'Denman'`

We search CMR with **no spatial filter** (all granules — there's only one comprehensive v2 geopackage in this collection) and let the `Glac_Name` attribute do the AOI selection. If the granule is already in the local cache, `earthaccess.download` re-uses it.

The geopackage schema includes `Glac_Name` (per-feature glacier name) and `Year` (int acquisition year), so we can group features by year without parsing filenames or dates.

In [ ]:
auth = earthaccess.login(persist=True)


def download_if_missing(results, local_path, ext):
    """Skip earthaccess.download entirely if every expected file is already
    cached locally. Otherwise download only the missing granules.

    *ext* is the extension used to pick the primary asset out of a granule's
    data_links (lowercase, including the dot — e.g. '.gpkg' or '.nc'). A
    granule that exposes no URL with this extension is skipped (e.g. a
    shapefile-only variant when we asked for '.gpkg').
    """
    local_path = Path(local_path)
    local_path.mkdir(exist_ok=True)

    cached_paths, needs = [], []
    skipped_no_ext = 0
    for r in results:
        links = []
        for access in ('external', 'direct'):
            try:
                links = r.data_links(access=access)
            except Exception:
                links = []
            if links:
                break
        urls = [u for u in links if u.lower().endswith(ext)]
        if not urls:
            skipped_no_ext += 1
            continue
        name = Path(urls[0].split('?')[0]).name
        cand = local_path / name
        if cand.exists() and cand.stat().st_size > 0:
            cached_paths.append(str(cand))
        else:
            needs.append(r)

    if cached_paths:
        print(f'  cache hit: {len(cached_paths)} file(s) in {local_path}')
    if skipped_no_ext:
        print(f'  skipped {skipped_no_ext} granule(s) with no *{ext} asset')
    fresh = []
    if needs:
        print(f'  downloading {len(needs)} missing granule(s)...')
        fresh = earthaccess.download(needs, local_path=str(local_path))
    return cached_paths + list(fresh)


gl_cache = Path('./insar_gl_cache')
# NO spatial bbox — morton is the spatial index; Glac_Name is the AOI filter.
gl_results = earthaccess.search_data(short_name='NSIDC-0498', version='2')
print(f'{len(gl_results)} NSIDC-0498 granules available')
gl_files = download_if_missing(gl_results, gl_cache, ext='.gpkg')
gpkg_files = [Path(f) for f in gl_files if str(f).lower().endswith('.gpkg')]
print(f'{len(gpkg_files)} .gpkg files on disk')

In [ ]:
GLACIER_NAME = 'Denman'

# Load every geopackage, concatenate, then filter by name. A normal workflow
# has one .gpkg with all features; this handles the general case.
gdfs = []
for p in sorted(gpkg_files):
    try:
        gdf = gpd.read_file(p)
    except Exception as e:
        print(f'skip {p.name}: {e}')
        continue
    if gdf.crs and gdf.crs.to_epsg() != 4326:
        gdf = gdf.to_crs(4326)
    gdfs.append(gdf)

if not gdfs:
    raise RuntimeError('no grounding-line geopackage could be read')
gdf_all = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True), crs='EPSG:4326')

# Filter by glacier name. The attribute is 'Glac_Name' in NSIDC-0498 v2.
gdf_denman = gdf_all[gdf_all['Glac_Name'] == GLACIER_NAME].copy()
print(f'{len(gdf_denman)} features with Glac_Name == {GLACIER_NAME!r}')

# Group by the 'Year' column (int32 in the schema).
gl_by_year: dict[int, gpd.GeoDataFrame] = {
    int(y): sub.copy() for y, sub in gdf_denman.groupby('Year')
}
GL_YEARS = sorted(gl_by_year.keys())
for y in GL_YEARS:
    print(f'  {y}: {len(gl_by_year[y])} features')

### Compute grounding-line morton coverage per year

For each year, we:
1. Unpack every `LineString` / `MultiLineString` into lat/lon arrays.
2. Call `mortie.linestring_coverage(..., order=17)` — one call, list-of-arrays output.
3. Union across all lines for that year.
4. Add the `morton_buffer(..., k=1)` ring.
5. Clip the final buffered set down to each match order (14 and 15) for intersection with velocity cells later.

In [ ]:
def gl_year_to_morton(gdf_year: gpd.GeoDataFrame) -> dict:
    lats_parts, lons_parts = [], []
    for geom in gdf_year.geometry:
        if geom is None or geom.is_empty:
            continue
        la, lo = shapely_to_latlon(geom)
        lats_parts.extend(la); lons_parts.extend(lo)

    per_line = mortie.linestring_coverage(lats_parts, lons_parts, order=GL_ORDER)
    cells17 = np.unique(np.concatenate(per_line)) if per_line else np.empty(0, np.int64)
    if cells17.size == 0:
        return {'lats_parts': lats_parts, 'lons_parts': lons_parts,
                'cells17': cells17,
                'cells17_buffered': cells17,
                'cells_clipped': {o: np.empty(0, np.int64) for o in MATCH_ORDERS}}

    ring = mortie.morton_buffer(cells17, k=GL_BUFFER_K)
    cells17_buffered = np.union1d(cells17, ring)
    cells_clipped = {
        o: np.unique(morton_clip_order(cells17_buffered, GL_ORDER, o))
        for o in MATCH_ORDERS
    }
    return {'lats_parts': lats_parts, 'lons_parts': lons_parts,
            'cells17': cells17,
            'cells17_buffered': cells17_buffered,
            'cells_clipped': cells_clipped}

GL_MORTON = {y: gl_year_to_morton(gl_by_year[y]) for y in GL_YEARS}
for y in GL_YEARS:
    s = GL_MORTON[y]
    clipped_sizes = {o: s['cells_clipped'][o].size for o in MATCH_ORDERS}
    print(f'  {y}: {s["cells17"].size:,} order-17 cells + {s["cells17_buffered"].size - s["cells17"].size:,} buffer ring '
          f'-> clipped {clipped_sizes}')

## 2. Velocity — NSIDC-0761 v1, four multi-year epochs

Four NetCDF-4 files (one per epoch), each a 450 m EPSG:3031 raster covering the full Antarctic continent. We download **all** of them (no CMR bbox), then compute morton indices per pixel at each match order in **row-chunks** so we never hold a full ~100M-pixel morton array at once. Within each chunk we immediately drop any pixel whose morton cell isn't in the union of all grounding-line years' clipped cells — that gets us from ~100M pixels to a few thousand in one pass. The sparse match set is what the slider renders.

In [ ]:
vel_cache = Path('./insar_vel_cache')

# NO spatial bbox — we pull the full continent-wide grids for each of the
# four multi-year epochs and let morton indexing localise to Denman later.
vel_results = earthaccess.search_data(short_name='NSIDC-0761', version='1')
print(f'{len(vel_results)} velocity granules returned')
vel_files = download_if_missing(vel_results, vel_cache, ext='.nc')
nc_files = sorted([Path(f) for f in vel_files if str(f).lower().endswith('.nc')])
for p in nc_files:
    print(f'  {p.name}  ({p.stat().st_size/1e6:.1f} MB)')

In [ ]:
# Only reason we need pyproj here: NSIDC-0761 pixels are in EPSG:3031 (x,y),
# and mortie.geo2mort wants lat/lon in EPSG:4326. No Denman bbox is involved.
to_4326 = Transformer.from_crs('EPSG:3031', 'EPSG:4326', always_xy=True)

In [ ]:
def _infer_var(ds, candidates):
    """Return the first variable name in *candidates* that exists in *ds* (case-insensitive)."""
    lowmap = {name.lower(): name for name in ds.data_vars}
    for c in candidates:
        if c.lower() in lowmap:
            return lowmap[c.lower()]
    return None


def _infer_epoch(ds, path: Path) -> tuple[int, int]:
    """Extract (start_year, end_year) from CF attributes or the filename."""
    for a, b in (('time_coverage_start', 'time_coverage_end'),
                 ('start_time', 'stop_time')):
        if a in ds.attrs and b in ds.attrs:
            sy = pd.to_datetime(ds.attrs[a]).year
            ey = pd.to_datetime(ds.attrs[b]).year
            return sy, ey
    import re
    ys = re.findall(r'(19|20)\d{2}', path.name)
    if len(ys) >= 2:
        return int(ys[0]), int(ys[1])
    return (-1, -1)


# Precompute the union of all GL years' clipped cells, per match order. Any
# velocity pixel whose morton isn't in this set is irrelevant for Denman — so
# we can drop it immediately during the row-chunk scan.
GL_UNION = {o: np.unique(np.concatenate(
    [GL_MORTON[y]['cells_clipped'][o] for y in GL_YEARS] or [np.empty(0, np.int64)]
)) for o in MATCH_ORDERS}
for o in MATCH_ORDERS:
    print(f'GL union at order {o}: {GL_UNION[o].size:,} cells')


# Denman x/y range in EPSG:3031 — used ONLY to load a small display window
# per epoch (for the faint background raster). Not used for morton matching.
_lons_bb = np.array([DENMAN_LONMIN, DENMAN_LONMAX, DENMAN_LONMAX, DENMAN_LONMIN])
_lats_bb = np.array([DENMAN_LATMIN, DENMAN_LATMIN, DENMAN_LATMAX, DENMAN_LATMAX])
_to_3031 = Transformer.from_crs('EPSG:4326', 'EPSG:3031', always_xy=True)
_xs, _ys = _to_3031.transform(_lons_bb, _lats_bb)
DENMAN_XMIN_3031, DENMAN_XMAX_3031 = float(min(_xs)), float(max(_xs))
DENMAN_YMIN_3031, DENMAN_YMAX_3031 = float(min(_ys)), float(max(_ys))
print(f'Denman display window (EPSG:3031): '
      f'x=[{DENMAN_XMIN_3031:.0f}, {DENMAN_XMAX_3031:.0f}], '
      f'y=[{DENMAN_YMIN_3031:.0f}, {DENMAN_YMAX_3031:.0f}]')

In [ ]:
def process_velocity_epoch(path: Path, chunk_rows: int = 1024) -> dict:
    """Open a NSIDC-0761 NetCDF, scan the full continent in row-chunks, and
    return the sparse set of pixels whose morton cell (at any match order)
    lies inside the union of Denman grounding-line cells.

    Also preloads a small Denman-window speed raster for the faint
    background layer at render time. No other bbox is applied.
    """
    ds = xr.open_dataset(path, engine='h5netcdf', chunks={})
    xname = 'x' if 'x' in ds.dims else 'X'
    yname = 'y' if 'y' in ds.dims else 'Y'

    vx_name = _infer_var(ds, ['VX', 'vx', 'VELOCITY_X'])
    vy_name = _infer_var(ds, ['VY', 'vy', 'VELOCITY_Y'])
    v_name  = _infer_var(ds, ['V',  'v',  'VV', 'SPEED'])

    def _speed_row_block(sub):
        if v_name is not None:
            a = sub[v_name].values.astype(np.float32)
        else:
            vx = sub[vx_name].values.astype(np.float32)
            vy = sub[vy_name].values.astype(np.float32)
            a = np.sqrt(vx * vx + vy * vy)
        while a.ndim > 2:
            a = a.squeeze(axis=0)
        return a

    xs_full = ds[xname].values.astype(np.float64)
    ys_full = ds[yname].values.astype(np.float64)
    start_year, end_year = _infer_epoch(ds, path)

    # --- chunked full-continent scan ---------------------------------------
    matched = {o: {'x': [], 'y': [], 'speed': [], 'morton': []} for o in MATCH_ORDERS}
    n_scanned = 0

    for y0 in range(0, ys_full.size, chunk_rows):
        y1 = min(y0 + chunk_rows, ys_full.size)
        sub = ds.isel({yname: slice(y0, y1)})
        speed_block = _speed_row_block(sub)     # (rows, cols) float32
        ys_block = ys_full[y0:y1]
        xs = xs_full
        xx, yy = np.meshgrid(xs, ys_block)
        lons, lats = to_4326.transform(xx.ravel(), yy.ravel())

        for o in MATCH_ORDERS:
            mort = mortie.geo2mort(np.ascontiguousarray(lats),
                                   np.ascontiguousarray(lons),
                                   order=o).reshape(xx.shape)
            hit = np.isin(mort, GL_UNION[o])
            if not hit.any():
                continue
            # record the matches
            rr, cc = np.nonzero(hit)
            matched[o]['x'].append(xx[rr, cc])
            matched[o]['y'].append(yy[rr, cc])
            matched[o]['speed'].append(speed_block[rr, cc])
            matched[o]['morton'].append(mort[rr, cc])

        n_scanned += speed_block.size

    matched_final = {}
    for o in MATCH_ORDERS:
        if matched[o]['x']:
            matched_final[o] = {
                'x': np.concatenate(matched[o]['x']),
                'y': np.concatenate(matched[o]['y']),
                'speed': np.concatenate(matched[o]['speed']),
                'morton': np.concatenate(matched[o]['morton']),
            }
        else:
            matched_final[o] = {k: np.empty(0, np.float32) for k in ('x', 'y', 'speed')}
            matched_final[o]['morton'] = np.empty(0, np.int64)

    # --- small Denman-window background raster (display only) -------------
    if ys_full[0] > ys_full[-1]:
        ysel = slice(DENMAN_YMAX_3031, DENMAN_YMIN_3031)
    else:
        ysel = slice(DENMAN_YMIN_3031, DENMAN_YMAX_3031)
    xsel = slice(DENMAN_XMIN_3031, DENMAN_XMAX_3031)
    ds_win = ds.sel({xname: xsel, yname: ysel})
    bg_speed = _speed_row_block(ds_win)
    bg_xs = ds_win[xname].values.astype(np.float64)
    bg_ys = ds_win[yname].values.astype(np.float64)

    ds.close()

    return {
        'start_year': start_year, 'end_year': end_year,
        'path': path, 'n_scanned': n_scanned,
        'matched': matched_final,
        'bg_x': bg_xs, 'bg_y': bg_ys, 'bg_speed': bg_speed,
    }


import time
VEL_EPOCHS: list[dict] = []
for p in nc_files:
    t0 = time.perf_counter()
    try:
        ep = process_velocity_epoch(p)
    except Exception as e:
        print(f'{p.name} failed: {e}')
        continue
    dt = time.perf_counter() - t0
    counts = {o: ep['matched'][o]['morton'].size for o in MATCH_ORDERS}
    print(f'  {p.name}  [{ep["start_year"]}–{ep["end_year"]}]  '
          f'scanned {ep["n_scanned"]:,} px in {dt:.1f} s, '
          f'matched {counts}, bg window {ep["bg_speed"].shape}')
    VEL_EPOCHS.append(ep)

VEL_EPOCHS.sort(key=lambda e: (e['start_year'], e['end_year']))

## 3. Match grounding-line year → velocity epoch

For a given GL year, pick the velocity epoch whose range contains it; if none, pick the nearest by midpoint.

In [ ]:
def match_epoch(year: int) -> dict:
    for ep in VEL_EPOCHS:
        if ep['start_year'] <= year <= ep['end_year']:
            return ep
    return min(VEL_EPOCHS,
              key=lambda ep: abs(year - (ep['start_year'] + ep['end_year']) / 2))

for y in GL_YEARS:
    m = match_epoch(y)
    print(f'  GL {y} -> velocity {m["start_year"]}–{m["end_year"]}')

## 4. Interactive slider

Two panels — one per match order — each showing:
- the Denman-window speed raster (faint, cropped *only* for display),
- the **matched pixels** for the selected year, coloured by speed,
- the raw grounding line overlaid in red.

The slider picks a grounding-line year. On change, we intersect that year's order-N clipped cells with the epoch's sparse matched pixel set (precomputed) and re-scatter.

In [ ]:
SPEED_VMIN, SPEED_VMAX = 0.0, 1500.0
SPEED_CMAP = plt.get_cmap('viridis').copy()
SPEED_CMAP.set_bad('#dcdcdc')

# Scatter marker size per match order (cartopy scatter marker area, points^2).
# Order 15 has ~200 m cells vs 450 m pixels — tiny dots; order 14 is larger.
MARKER_SIZE = {14: 18, 15: 8}


def render(year: int, fig=None, axes=None):
    gl = GL_MORTON[year]
    ep = match_epoch(year)
    if fig is None:
        fig, axes = plt.subplots(1, len(MATCH_ORDERS),
                                 figsize=(7.5 * len(MATCH_ORDERS), 7),
                                 subplot_kw={'projection': ccrs.SouthPolarStereo()})
    else:
        for ax in axes: ax.clear()

    # Faint background (Denman window)
    bg_speed = np.where(np.isfinite(ep['bg_speed']) & (ep['bg_speed'] > 0),
                        ep['bg_speed'], np.nan)
    bg_extent = (ep['bg_x'].min(), ep['bg_x'].max(),
                 ep['bg_y'].min(), ep['bg_y'].max())
    bg_origin = 'upper' if ep['bg_y'][0] > ep['bg_y'][-1] else 'lower'

    for ax, order in zip(axes, MATCH_ORDERS):
        ax.set_extent([DENMAN_LONMIN, DENMAN_LONMAX,
                       DENMAN_LATMIN, DENMAN_LATMAX], ccrs.PlateCarree())
        ax.add_feature(cfeature.LAND, facecolor='#f1efe8')
        ax.coastlines(resolution='50m', lw=0.4, color='#888')

        ax.imshow(
            bg_speed, origin=bg_origin, extent=bg_extent,
            transform=ccrs.epsg(3031),
            cmap=SPEED_CMAP, vmin=SPEED_VMIN, vmax=SPEED_VMAX,
            alpha=0.35, interpolation='nearest',
        )

        # Intersect this year's clipped cells with the epoch's sparse matches
        year_cells = gl['cells_clipped'][order]
        m_data = ep['matched'][order]
        if m_data['morton'].size and year_cells.size:
            hit = np.isin(m_data['morton'], year_cells)
            ax.scatter(
                m_data['x'][hit], m_data['y'][hit],
                c=m_data['speed'][hit],
                cmap=SPEED_CMAP, vmin=SPEED_VMIN, vmax=SPEED_VMAX,
                s=MARKER_SIZE[order], marker='s',
                edgecolors='none', transform=ccrs.epsg(3031),
            )
            n_match = int(hit.sum())
        else:
            n_match = 0

        # Grounding line in red
        for la, lo in zip(gl['lats_parts'], gl['lons_parts']):
            ax.plot(lo, la, color='crimson', lw=1.3, transform=ccrs.PlateCarree())

        ax.set_title(
            f'match order {order} ({cell_width_m(order):.0f} m cells)\n'
            f'GL {year}   ×   vel {ep["start_year"]}–{ep["end_year"]}   '
            f'→   {n_match:,} matched pixels',
            fontsize=11,
        )

    return fig, axes

In [ ]:
%matplotlib inline
# Draw one initial figure, then hook up an ipywidgets slider that redraws it.
initial_year = GL_YEARS[len(GL_YEARS) // 2]
fig, axes = render(initial_year)
plt.close(fig)  # prevent double-display — the widget will show it

out = widgets.Output()
slider = widgets.SelectionSlider(
    options=GL_YEARS,
    value=initial_year,
    description='GL year',
    continuous_update=False,
    layout=widgets.Layout(width='85%'),
)

def _on_change(change):
    with out:
        out.clear_output(wait=True)
        fig, axes = render(change['new'])
        plt.show()

slider.observe(_on_change, names='value')
# Seed the initial view
with out:
    fig, axes = render(initial_year)
    plt.show()

display(slider, out)

### Notes

- The left panel (order 14, ~400 m) aggregates groups of 4 order-15 cells. Each highlighted pixel is the full order-14 tile containing the grounding line — coarser, but visually simpler.
- The right panel (order 15, ~200 m) is close to the native velocity pixel size (450 m per side, one velocity pixel = 4 order-15 cells), which gives a tight hug around the grounding line.
- The morton-indexing approach scales beyond this notebook: once you have morton indices on both sides, any combination of `np.isin` / `np.intersect1d` / `np.union1d` lets you ask questions like *"which velocity pixels intersect any of these polygons, these flight lines, and these tracks?"* without writing a bespoke spatial join for each pair.